In [2]:
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Create a Vector Store from Text

In [3]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

documents = [

    Document(
        page_content="Python is a programming language."
    ),

    Document(
        page_content="LangChain is an LLM framework."
    ),

    Document(
        page_content="Machine Learning is a subset of AI."
    )

]

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

print(vectorstore)

C:\Users\Hp\AppData\Local\Temp\ipykernel_10132\2427788644.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

# Similarity Search

In [4]:
results = vectorstore.similarity_search(
    "What is LangChain?"
)

for doc in results:
    print(doc.page_content)

NameError: name 'vectorstore' is not defined

# Top-K Search

In [ ]:
results = vectorstore.similarity_search(
    "Artificial Intelligence",
    k=2
)

for doc in results:
    print(doc.page_content)

# Similarity Search with Score

In [ ]:
results = vectorstore.similarity_search_with_score(
    "Python"
)

for document, score in results:

    print(document.page_content)
    print(score)

# Add More Documents

In [ ]:
from langchain_core.documents import Document

new_docs = [

    Document(
        page_content="Deep Learning uses neural networks."
    ),

    Document(
        page_content="RAG combines retrieval with LLMs."
    )

]

vectorstore.add_documents(new_docs)

print("Documents Added")

# Delete Documents

In [ ]:
ids = vectorstore.index_to_docstore_id.values()

vectorstore.delete(
    ids=list(ids)[:1]
)

print("Deleted Successfully")

# Save Vector Store

In [ ]:
vectorstore.save_local("faiss_index")

# Load Vector Store

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(vectorstore)

# Convert into Retriever

In [ ]:
retriever = vectorstore.as_retriever()

results = retriever.invoke(
    "What is LangChain?"
)

for doc in results:
    print(doc.page_content)

# Retriever with Search Parameters

In [ ]:
retriever = vectorstore.as_retriever(

    search_kwargs={
        "k":2
    }

)

documents = retriever.invoke(
    "Artificial Intelligence"
)

for doc in documents:
    print(doc.page_content)

# Create Vector Store from PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

loader = PyPDFLoader("AI.pdf")

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print(vectorstore)

# Metadata Filtering

In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS

documents = [

    Document(
        page_content="Python Tutorial",
        metadata={"category":"Programming"}
    ),

    Document(
        page_content="AI Notes",
        metadata={"category":"Artificial Intelligence"}
    )

]

vectorstore = FAISS.from_documents(
    documents,
    embeddings
)

results = vectorstore.similarity_search(
    "Python"
)

for doc in results:

    print(doc.page_content)
    print(doc.metadata)

# Retrieve and Ask the LLM

In [ ]:
documents = retriever.invoke(
    "Explain Machine Learning"
)

context = "\n".join(
    doc.page_content
    for doc in documents
)

prompt = f"""
Answer using only the context.

Context:
{context}

Question:
What is Machine Learning?
"""

response = llm.invoke(prompt)

print(response.content)